<a href="https://colab.research.google.com/github/AnasMardood/Process_Mining_Project/blob/main/S%C3%BCre%C3%A7_Madencili%C4%9Fi_Projesi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install streamlit pyngrok pandas matplotlib seaborn plotly


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 69.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 6.1 MB/s eta 0:00:00


In [ ]:
from pyngrok import ngrok
from pyngrok.conf import PyngrokConfig  # Daha iyi konfigürasyon için

In [ ]:
# Daha iyi yöntem:
import os
from google.colab import userdata

try:
    NGROK_TOKEN = userdata.get("KEY")  # Colab'ın secret manager'ından al
    ngrok.set_auth_token(NGROK_TOKEN)
except Exception as e:
    print("Token hatası:", e)

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import plotly.express as px

# Sayfa konfigürasyonu
st.set_page_config(
    page_title="Süreç Madenciliği Analiz",
    page_icon="📊",
    layout="wide"
)

def load_data(uploaded_file):
    """CSV dosyasını yükler ve temizler"""
    try:
        df = pd.read_csv(uploaded_file)

        # Gerekli sütun kontrolü
        required_columns = ['Case ID', 'Activity Name', 'Start Time', 'End Time']
        if not all(col in df.columns for col in required_columns):
            st.error(f"CSV dosyası şu sütunları içermeli: {', '.join(required_columns)}")
            return None

        # Tarih dönüşümü
        df['Start Time'] = pd.to_datetime(df['Start Time'])
        df['End Time'] = pd.to_datetime(df['End Time'])

        # Süre hesaplama
        df['Duration'] = df['End Time'] - df['Start Time']

        return df
    except Exception as e:
        st.error(f"Dosya okuma hatası: {e}")
        return None

def analyze_process(df):
    """Temel analizleri yapar"""
    if df is None:
        return None

    analysis = {}

    # Case süreleri
    case_durations = df.groupby('Case ID')['Duration'].sum()
    analysis['case_durations'] = case_durations

    # Aktivite frekansları
    activity_counts = df['Activity Name'].value_counts()
    analysis['activity_counts'] = activity_counts

    # Ortalama süreç süresi
    analysis['avg_duration'] = case_durations.mean()

    # Geçişler
    df_sorted = df.sort_values(['Case ID', 'Start Time'])
    df_sorted['Next Activity'] = df_sorted.groupby('Case ID')['Activity Name'].shift(-1)
    transitions = df_sorted.dropna().groupby(['Activity Name', 'Next Activity']).size().reset_index(name='Count')
    analysis['transitions'] = transitions.sort_values('Count', ascending=False)

    return analysis

def main():
    st.title("📊 Süreç Madenciliği Analiz Aracı")

    # Dosya yükleme
    uploaded_file = st.file_uploader("CSV dosyası yükleyin", type="csv")

    if uploaded_file is not None:
        df = load_data(uploaded_file)

        if df is not None:
            st.success("Veri başarıyla yüklendi!")
            st.write("### Veri Önizleme", df.head(20))

            analysis = analyze_process(df)

            # Analiz sonuçları
            st.write("## Analiz Sonuçları")

            col1, col2 = st.columns(2)

            with col1:
                st.write("### Aktivite Frekansları")
                fig1 = px.bar(analysis['activity_counts'],
                              x=analysis['activity_counts'].index,
                              y=analysis['activity_counts'].values,
                              labels={'x':'Aktivite', 'y':'Frekans'})
                st.plotly_chart(fig1, use_container_width=True)

            with col2:
                st.write("### Case Süreleri")
                fig2 = px.histogram(analysis['case_durations'].dt.total_seconds()/3600,
                                   labels={'value':'Süre (saat)'},
                                   title='Case Süreleri Dağılımı')
                st.plotly_chart(fig2, use_container_width=True)

            st.write(f"### Ortalama Süreç Tamamlama Süresi: {analysis['avg_duration'].total_seconds()/3600:.2f} saat")

            st.write("### En Sık Geçişler")
            st.dataframe(analysis['transitions'].head(10))

            # İleri seviye görselleştirme
            st.write("## Süreç Akışı")
            try:
                from graphviz import Digraph
                dot = Digraph()

                top_transitions = analysis['transitions'].head(10)
                for _, row in top_transitions.iterrows():
                    dot.edge(row['Activity Name'], row['Next Activity'], label=str(row['Count']))

                st.graphviz_chart(dot)
            except Exception as e:
                st.warning(f"Graphviz hatası: {e}. Akış diyagramı gösterilemiyor.")

if __name__ == "__main__":
    main()

Overwriting app.py


In [ ]:
from pyngrok import ngrok
from pyngrok.conf import PyngrokConfig
from IPython import get_ipython

try:
    # Tunnel oluşturma
    public_url = ngrok.connect(
        addr=8501,
        proto="http",
        pyngrok_config=PyngrokConfig(region="eu")  # Artık tanımlı
    )

    print(f"Uygulamanız şu adreste erişilebilir: {public_url}")
    get_ipython().system('streamlit run app.py --server.port 8501 --server.headless true &>/dev/null &')

except Exception as e:
    print("Hata:", e)

Uygulamanız şu adreste erişilebilir: NgrokTunnel: "https://ee22-35-231-27-228.ngrok-free.app" -> "http://localhost:8501"


In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

# Aktivite isimleri
activities = [
    "Sipariş Alındı",
    "Ödeme Onayı",
    "Stok Kontrolü",
    "Kargoya Hazırlık",
    "Kargoya Verildi",
    "Teslim Edildi",
    "İade Başvurusu",
    "İade Onayı",
    "İade Alındı",
    "İade Tamamlandı"
]

# Rastgele tarih üretme fonksiyonu
def random_date(start_date, end_date):
    time_between = end_date - start_date
    random_days = random.randrange(time_between.days)
    return start_date + timedelta(days=random_days)

# Veri üretme
def generate_process_data(num_cases=100, max_events_per_case=8):
    data = []
    start_date = datetime(2023, 1, 1)
    end_date = datetime(2023, 12, 31)

    for case_id in range(1, num_cases + 1):
        num_events = random.randint(3, max_events_per_case)
        case_activities = random.sample(activities, num_events)

        # Süreç akışını daha gerçekçi yapmak için bazı aktiviteleri sıralı yapalım
        if "Sipariş Alındı" in case_activities:
            case_activities.remove("Sipariş Alındı")
            case_activities.insert(0, "Sipariş Alındı")
        if "Teslim Edildi" in case_activities:
            case_activities.remove("Teslim Edildi")
            case_activities.append("Teslim Edildi")

        prev_end_time = None
        for i, activity in enumerate(case_activities):
            if prev_end_time is None:
                start_time = random_date(start_date, end_date)
            else:
                start_time = prev_end_time + timedelta(hours=random.randint(1, 72))

            duration = timedelta(hours=random.randint(1, 48))
            end_time = start_time + duration

            data.append({
                "Case ID": f"CASE-{case_id:03d}",
                "Activity Name": activity,
                "Start Time": start_time.strftime("%Y-%m-%d %H:%M:%S"),
                "End Time": end_time.strftime("%Y-%m-%d %H:%M:%S")
            })

            prev_end_time = end_time

    return pd.DataFrame(data)

# 150 case ve case başına 3-8 aktivite ile veri üret
df = generate_process_data(num_cases=150, max_events_per_case=8)

# CSV olarak kaydet
csv_filename = "process_mining_data.csv"
df.to_csv(csv_filename, index=False)

print(f"{len(df)} kayıt ile '{csv_filename}' dosyası oluşturuldu.")
print("Örnek veri:")
print(df.head(10))

823 kayıt ile 'process_mining_data.csv' dosyası oluşturuldu.
Örnek veri:
    Case ID     Activity Name           Start Time             End Time
0  CASE-001   İade Tamamlandı  2023-09-24 00:00:00  2023-09-25 05:00:00
1  CASE-001        İade Onayı  2023-09-25 14:00:00  2023-09-26 02:00:00
2  CASE-001   Kargoya Verildi  2023-09-26 15:00:00  2023-09-27 18:00:00
3  CASE-001       İade Alındı  2023-09-27 20:00:00  2023-09-28 21:00:00
4  CASE-002    İade Başvurusu  2023-06-05 00:00:00  2023-06-05 09:00:00
5  CASE-002       Ödeme Onayı  2023-06-07 17:00:00  2023-06-09 13:00:00
6  CASE-002  Kargoya Hazırlık  2023-06-10 13:00:00  2023-06-11 18:00:00
7  CASE-002       İade Alındı  2023-06-13 19:00:00  2023-06-15 05:00:00
8  CASE-002   Kargoya Verildi  2023-06-16 04:00:00  2023-06-18 03:00:00
9  CASE-002     Teslim Edildi  2023-06-21 01:00:00  2023-06-21 23:00:00
